In [1]:
# Get the library files
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
dataset = pd.read_csv("DillibabuSarva_DefectDataset.csv")

In [3]:
# Display the dataset
dataset

,SHA,cbo,wmc,dit,rfc,lcom,totalMethods,totalFields,nosi,loc,...,tryCatchQty,parenthesizedExpsQty,stringLiteralsQty,numbersQty,assignmentsQty,mathOperationsQty,variablesQty,maxNestedBlocks,uniqueWordsQty,defect
0,7a955fd6c7de2bd912be544dcfe77f9173a7aa600,5,60,2,55,189,27,5,30,247,...,4,2,47,9,27,5,17,3,191,0
1,000f1ab4780fc9460975791c52597f7c04e15be70,3,10,1,1,9,7,4,1,38,...,0,0,0,22,4,0,4,2,69,0
2,000f1ab4780fc9460975791c52597f7c04e15be71,3,10,1,1,9,7,4,0,38,...,0,0,0,22,4,0,4,2,69,1
3,0024dbdd6ba3cc7797cc0b1ae537dcdc488c4c270,20,59,3,63,189,24,9,4,262,...,0,6,6,14,45,8,41,4,222,0
4,0024dbdd6ba3cc7797cc0b1ae537dcdc488c4c271,21,58,2,61,189,24,9,0,260,...,0,6,6,14,45,8,41,4,222,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6047,ffd1ed788cbf10bed00d49d79c7ee44250c36ac11,52,124,12,144,963,110,9,0,804,...,0,0,26,16,32,4,30,6,689,1
6048,ffdf4a3fcccb7489548c6a2ff6af7cebc92a180c0,24,27,2,46,4,8,8,0,126,...,0,1,3,14,27,0,24,3,108,0
6049,ffdf4a3fcccb7489548c6a2ff6af7cebc92a180c1,22,27,1,46,4,8,8,0,126,...,0,1,3,14,27,0,24,3,108,1
6050,ffe7c9989a4553d35fd1d5041d0cece0a673a0c80,3,12,2,12,28,8,0,1,67,...,2,0,0,2,10,0,8,2,36,0


In [4]:
# Display the column names
dataset.columns

Index(['SHA', 'cbo', 'wmc', 'dit', 'rfc', 'lcom', 'totalMethods',
       'totalFields', 'nosi', 'loc', 'returnQty', 'loopQty', 'comparisonsQty',
       'tryCatchQty', 'parenthesizedExpsQty', 'stringLiteralsQty',
       'numbersQty', 'assignmentsQty', 'mathOperationsQty', 'variablesQty',
       'maxNestedBlocks', 'uniqueWordsQty', 'defect'],
      dtype='object')

In [5]:
# Feature selection result
selected_features = ['nosi','dit','cbo','rfc','maxNestedBlocks',
                     'uniqueWordsQty','assignmentsQty','numbersQty',
                     'tryCatchQty','parenthesizedExpsQty']

X = dataset[selected_features]
y = dataset['defect']

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [7]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators =10, criterion = 'entropy', random_state =0)
classifier.fit(X_train, y_train)

RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)

In [8]:
y_pred = classifier.predict(X_test)

In [9]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

In [10]:
print(cm)

[[644 264]
 [285 623]]


In [11]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred)

In [12]:
# AdaBoostClassification Report
print(clf_report)

              precision    recall  f1-score   support

           0       0.69      0.71      0.70       908
           1       0.70      0.69      0.69       908

    accuracy                           0.70      1816
   macro avg       0.70      0.70      0.70      1816
weighted avg       0.70      0.70      0.70      1816



In [13]:
# Finding the outliers for our dataset
Q1 = dataset[selected_features].quantile(0.25)
Q3 = dataset[selected_features].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR
#Check
for col in selected_features:
    less = (dataset[col] < lower[col]).sum()
    great = (dataset[col] > upper[col]).sum()
    print(f"{col}: Lesser = {less}, Greater = {great}")

nosi: Lesser = 0, Greater = 967
dit: Lesser = 0, Greater = 788
cbo: Lesser = 0, Greater = 403
rfc: Lesser = 0, Greater = 367
maxNestedBlocks: Lesser = 0, Greater = 411
uniqueWordsQty: Lesser = 0, Greater = 431
assignmentsQty: Lesser = 0, Greater = 490
numbersQty: Lesser = 0, Greater = 651
tryCatchQty: Lesser = 0, Greater = 651
parenthesizedExpsQty: Lesser = 0, Greater = 671


In [14]:
# Replacing the outliers with mean values
for col in selected_features:
    mean = dataset[col].mean()
    dataset[col] = np.where((dataset[col] > upper[col]) | (dataset[col] < lower[col]), mean, dataset[col])

In [15]:
# After replacing the outliers
for col in selected_features:
    less = (dataset[col] < lower[col]).sum()
    great = (dataset[col] > upper[col]).sum()
    print(f"{col}: Lesser = {less}, Greater = {great}")

nosi: Lesser = 0, Greater = 0
dit: Lesser = 0, Greater = 0
cbo: Lesser = 0, Greater = 0
rfc: Lesser = 0, Greater = 0
maxNestedBlocks: Lesser = 0, Greater = 0
uniqueWordsQty: Lesser = 0, Greater = 0
assignmentsQty: Lesser = 0, Greater = 0
numbersQty: Lesser = 0, Greater = 0
tryCatchQty: Lesser = 0, Greater = 0
parenthesizedExpsQty: Lesser = 0, Greater = 0


In [16]:
A = dataset[selected_features]
b = dataset['defect']
A_train, A_test, b_train, b_test = train_test_split(A, b, test_size = 0.3, random_state = 42, stratify = b)

In [18]:
classifier_recheck = RandomForestClassifier(n_estimators =10, criterion = 'entropy', random_state =0)
# fitting the model for grid search
classifier_recheck.fit(A_train, b_train)

RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)

In [19]:
b_pred = classifier_recheck.predict(A_test)

In [20]:
cmodel = confusion_matrix(b_test,b_pred)
print(cmodel)

[[649 259]
 [276 632]]


In [21]:
# AdaBoostClassification Report after replacing outliers
clf_report_check = classification_report(b_test, b_pred)
print(clf_report_check)

              precision    recall  f1-score   support

           0       0.70      0.71      0.71       908
           1       0.71      0.70      0.70       908

    accuracy                           0.71      1816
   macro avg       0.71      0.71      0.71      1816
weighted avg       0.71      0.71      0.71      1816



In [23]:
# Here, we could see accuracy increased by 1%

In [24]:
from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV

In [25]:
# 1. Create a synthetic dataset
A, b = make_classification(n_samples=1000, n_features=20, random_state=42)
A_train, A_test, b_train, b_test = train_test_split(A, b, test_size=0.2, random_state=42)

In [27]:
rf = RandomForestClassifier(random_state=42)

In [31]:
# Define the parameter grid to explore
param_grid = {
    'criterion': ['gini', 'entropy'],  # Function to measure the quality of a split
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
# Set up and run the Grid Search with 5-fold cross-validation
grid_search_rf = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1,  # Uses all available CPU cores
    verbose=1
)

# Train the grid search model
grid_search_rf.fit(A_train, b_train)

# Fetch the results table
rf_results = pd.DataFrame(grid_search_rf.cv_results_)
important_cols = ['param_max_depth', 'param_min_samples_split', 'mean_test_score']

# Display the top 5 performing parameter combinations
display(rf_results[important_cols].sort_values(by='mean_test_score', ascending=False).head())

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,param_max_depth,param_min_samples_split,mean_test_score
23,None,5,0.8975
19,20,5,0.8975
11,None,5,0.8950
7,20,5,0.8950
3,10,5,0.8925


In [32]:
# Set up the Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=1
)

In [33]:
# Fit the grid search to the training data
grid_search.fit(A_train, b_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [10, 20, None],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5]},
             scoring='accuracy', verbose=1)

In [34]:
# Convert the cv_results_ dictionary into a pandas DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

In [36]:
# Filter and sort to see the best combinations at the top
important_columns = [
    'mean_test_score', 
    'std_test_score', 
    'mean_fit_time'
]

# View the top 5 performing parameter combinations
display(results_df[important_columns].sort_values(by='mean_test_score', ascending=False))

,mean_test_score,std_test_score,mean_fit_time
23,0.89750,0.026101,0.456694
19,0.89750,0.026101,0.476013
11,0.89500,0.023519,0.399726
7,0.89500,0.023519,0.434683
3,0.89250,0.021794,0.406661
5,0.89250,0.029155,0.414793
15,0.89250,0.027783,0.483539
9,0.89250,0.029155,0.408485
13,0.89250,0.024174,0.589824
4,0.89125,0.028940,0.423927


In [37]:
# Extract and evaluate results
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")

Best Hyperparameters: {'criterion': 'entropy', 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5}
Best Cross-Validation Score: 0.8975


In [38]:
# Evaluate performance on the untouched test set
best_model = grid_search.best_estimator_
test_accuracy = best_model.score(A_test, b_test)
print(f"Test Set Accuracy: {test_accuracy:.4f}")

Test Set Accuracy: 0.8900


In [39]:
#importing pickle library for deployment phase
import pickle

In [40]:
#here we are assigning the saving model file name with extension to the filename variable
filename = "finalized-model_RF_Classification_Defect_Prediction.sav"
#using pickle dump method we are writing the file on disk
pickle.dump(classifier, open(filename, 'wb'))

In [41]:
import warnings
warnings.filterwarnings("ignore")
with open("finalized-model_RF_Classification_Defect_Prediction.sav", "rb") as f:
    model = pickle.load(f)
new_prediction = model.predict([[10, 5, 20, 70, 5, 200, 60, 30, 3, 10]])
print(f"Defect? {new_prediction[0]}")

Defect? 0


In [42]:
# Here after removing outliers, we could see accuracy we are getting as 89%